In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/+8-HQ-ipc2-B_opt_magres_new.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return np.round(a, 2), np.round(b,2), np.round(g,2)
    

In [5]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

11B1 sigma:
 -2.11495249601435

11B2 sigma:
 -2.1149524960142014

11B3 sigma:
 -2.11495249601438

11B4 sigma:
 -2.1149524960142254



In [6]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()


11B1 sigma:
 [[90.59389941 -4.0476885   1.71557026]
 [ 2.10559503 84.49606461 -6.70360403]
 [-4.05241751 -2.21612198 77.0268843 ]]

11B2 sigma:
 [[90.59389941  4.0476885   1.71557026]
 [-2.10559503 84.49606461  6.70360403]
 [-4.05241751  2.21612198 77.0268843 ]]

11B3 sigma:
 [[90.59389941 -4.0476885   1.71557026]
 [ 2.10559503 84.49606461 -6.70360403]
 [-4.05241751 -2.21612198 77.0268843 ]]

11B4 sigma:
 [[90.59389941  4.0476885   1.71557026]
 [-2.10559503 84.49606461  6.70360403]
 [-4.05241751  2.21612198 77.0268843 ]]



In [7]:
efg_tensor = atoms.species('B')[1].efg.V  # Extract the EFG tensor as a numpy array
print(efg_tensor)

# efg[0,0] = -0.0286; efg[0,1] = -0.1871 ; efg[0,2] = 0.1115;
# efg[1,0] = efg[0,1]; efg[1,1] = -0.0062; efg[1,2] = -0.0266;
# efg[2,0] = efg[0,2]; efg[2,1] = efg[1,2]; efg[2,2] = 0.0347;

[[-0.02857321  0.18707367  0.11149955]
 [ 0.18707367 -0.00615497  0.02661956]
 [ 0.11149955  0.02661956  0.03472818]]


In [8]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

CS_total[:,:] = atoms.species('B').ms.sigma[0]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('B')[0].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.04059
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-0.273 -1.784  1.063]
 [-1.784 -0.059 -0.254]
 [ 1.063 -0.254  0.331]]

CS Tensor:
 [[90.594 -4.048  1.716]
 [ 2.106 84.496 -6.704]
 [-4.052 -2.216 77.027]]

CS isotropic Tensor:
 [[84.039  0.     0.   ]
 [ 0.    84.039  0.   ]
 [ 0.     0.    84.039]]

CS symmetric Tensor:
 [[90.594 -0.971 -1.168]
 [-0.971 84.496 -4.46 ]
 [-1.168 -4.46  77.027]]

CS antisymmetric Tensor:
 [[ 0.    -3.077  2.884]
 [ 3.077  0.    -2.244]
 [-2.884  2.244  0.   ]]


In [9]:
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

 Unsorted Eigenvalues:
 [ 2.11456425e+00 -2.11495177e+00  3.87518944e-04] 

 Unsorted Eigenvectors:
 [[-0.65285638  0.74498025 -0.13705095]
 [ 0.59127767  0.61428871  0.52253239]
 [-0.47346516 -0.26010344  0.8415325 ]] 

Sorted Eigenvalues: 
 [ 3.87518944e-04  2.11456425e+00 -2.11495177e+00] 

Sorted Eigenvectors: 
 [[-0.13705095 -0.65285638  0.74498025]
 [ 0.52253239  0.59127767  0.61428871]
 [ 0.8415325  -0.47346516 -0.26010344]] 


 Unsorted Eigenvalues:
 [74.80756399 90.7658873  86.54339705] 

 Unsorted Eigenvectors:
 [[-0.09276105 -0.99155633 -0.09061694]
 [-0.42400323  0.12168251 -0.89744895]
 [-0.90089769  0.04482643  0.43171049]] 

Sorted Eigenvalues: 
 [86.54339705 90.7658873  74.80756399] 

Sorted Eigenvectors: 
 [[-0.09061694 -0.99155633 -0.09276105]
 [-0.89744895  0.12168251 -0.42400323]
 [ 0.43171049  0.04482643 -0.90089769]] 



In [10]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 0.000387518944374171 2.1145642524523143 -2.1149517713967847
CSA Tensor Components δyy, δxx, δzz: 
 86.54339704579223 90.76588729524553 74.80756399039382


In [11]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Fit Value']))

Qauntity        Fit Value
------------  -----------
CQ (MHz)        -2.11495
etaq             0.999634
iso_cs (ppm)    84.0389
csa (ppm)       -9.23139
etas             0.457406


In [12]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal for D-HQ:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal for D-HQ:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.65285638 -0.13705095  0.74498025]
 [ 0.59127767  0.52253239  0.61428871]
 [-0.47346516  0.8415325  -0.26010344]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal for D-HQ:
-60.64 105.08 -39.51 

Direction cosine csa: 

[[-0.99155633 -0.09061694 -0.09276105]
 [ 0.12168251 -0.89744895 -0.42400323]
 [ 0.04482643  0.43171049 -0.90089769]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal for D-HQ:
84.07 154.28 -77.66 



In [13]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 47.26 chi: 95.47 xi: 76.26 



**Rotation of tensors Crystal--> Tenon Frame**

In [14]:
#Euler angles Crystal--> Tenon Frame for RHQ
#Angles from Xray analysis
alpha = 153.4
beta = 149.3
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)

CSA Tensor in Crystal Frame: 
 [[90.59389941 -0.97104674 -1.16842362]
 [-0.97104674 84.49606461 -4.459863  ]
 [-1.16842362 -4.459863   77.0268843 ]]
CSA Tensor in Tenon Frame: 
 [[88.21566225  1.24927247 -4.33042579]
 [ 1.24927247 82.31214801  5.90284045]
 [-4.33042579  5.90284045 81.58903807]]
Quad Tensor in Crystal Frame: 
 [[-0.27250895 -1.78416222  1.06339545]
 [-1.78416222 -0.05870126 -0.25387649]
 [ 1.06339545 -0.25387649  0.33121021]]
Quad Tensor in Tenon Frame: 
 [[ 1.97482974  0.64541432  0.38201892]
 [ 0.64541432 -1.21503499 -0.96588207]
 [ 0.38201892 -0.96588207 -0.75979475]]
